<a href="https://colab.research.google.com/github/Kongbeng-21/SuperAi/blob/main/Heart_Disease_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/sample_submission.csv
/kaggle/input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/train.csv
/kaggle/input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/test.csv


# Setup

In [ ]:
print("✓ Setup เสร็จแล้ว")

✓ Setup เสร็จแล้ว


# Import and Config

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import fbeta_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_PATH = Path("/kaggle/working/submission.csv")
TARGET = "History of HeartDisease or Attack"
ID_COL = "ID"

print("✓ Import และ Config เสร็จแล้ว")


✓ Import และ Config เสร็จแล้ว


# Load Data

In [ ]:
def find_file(filename: str):
    for p in INPUT_ROOT.rglob(filename):
        return p
    return None

train_path = find_file("train.csv")
test_path = find_file("test.csv")
sample_sub_path = find_file("sample_submission.csv")

print("train_path =", train_path)
print("test_path =", test_path)
print("sample_sub_path =", sample_sub_path)

if train_path is None or test_path is None or sample_sub_path is None:
    raise FileNotFoundError("หา train.csv / test.csv / sample_submission.csv ไม่เจอใน /kaggle/input")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample submission shape:", sample_sub.shape)

display(train.head())


train_path = /kaggle/input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/train.csv
test_path = /kaggle/input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/test.csv
sample_sub_path = /kaggle/input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/sample_submission.csv
train shape: (223084, 20)
test shape: (74361, 19)
sample submission shape: (74361, 2)


,ID,History of HeartDisease or Attack,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,train_000001,No,Yes,Yes,Yes,40.68,Yes,No,No,No,No,Yes,No,Very Poor,Yes,Female,High school graduate,"$15,000 to less than $20,000",64,Yes
1,train_000002,No,No,No,No,24.36,Yes,No,No,Yes,No,No,Yes,Fair,No,Female,College graduate,"Less than $10,000",50,No
2,train_000003,No,Yes,Yes,Yes,27.33,No,No,No,No,No,Yes,Yes,Very Poor,Yes,Female,High school graduate,"$75,000 or more",61,Yes
3,train_000004,No,Yes,No,Yes,27.01,No,No,No,Yes,No,Yes,No,Good,No,Female,Some high school,"$35,000 to less than $50,000",74,Yes
4,train_000005,NaN,Yes,Yes,Yes,34.56,Yes,No,No,Yes,No,Yes,Yes,Very Poor,Yes,Male,Some high school,"$15,000 to less than $20,000",98,Yes


# Preprocess

In [ ]:
# ลบแถวที่ target ว่าง เพื่อไม่ให้ข้อมูลฝั่ง train ปนกับข้อมูลที่ label หาย
train = train[train[TARGET].notna()].copy()

# จัด target ให้อยู่ในรูปแบบไบนารี Yes=1, No=0
train[TARGET] = train[TARGET].astype(str).str.strip()
y = (train[TARGET] == "Yes").astype(int)

print(train[TARGET].value_counts(dropna=False))
print("positive rate =", y.mean())

display(train.isna().sum().sort_values(ascending=False).head(20))


History of HeartDisease or Attack
No     203322
Yes     18068
Name: count, dtype: int64
positive rate = 0.08161163557522923


Told High Cholesterol                     32043
Body Mass Index                           11725
Diagnosed Diabetes                            3
Difficulty Walking                            3
Doctor Visit Cost Barrier                     1
General Health                                1
Smoked 100+ Cigarettes                        1
ID                                            0
High Blood Pressure                           0
History of HeartDisease or Attack             0
Diagnosed Stroke                              0
Cholesterol Checked                           0
Health Care Coverage                          0
Heavy Alcohol Consumption                     0
Leisure Physical Activity                     0
Sex                                           0
Education Level                               0
Income Level                                  0
Age                                           0
Vegetable or Fruit Intake (1+ per Day)        0
dtype: int64

# Feature Engineering

In [ ]:
general_health_order = [
    "Very Poor",
    "Poor",
    "Fair",
    "Good",
    "Very Good",
    "Excellent"
]

education_order = [
    "Never attended school or only kindergarten",
    "Elementary",
    "Some high school",
    "High school graduate",
    "Some college or technical school",
    "College graduate"
]

income_order = [
    "Less than $10,000",
    "$10,000 to less than $15,000",
    "$15,000 to less than $20,000",
    "$20,000 to less than $25,000",
    "$25,000 to less than $35,000",
    "$35,000 to less than $50,000",
    "$50,000 to less than $75,000",
    "$75,000 or more"
]

def add_features(df):
    df = df.copy()

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace({"": np.nan, "nan": np.nan, "None": np.nan})

    df["Body Mass Index"] = pd.to_numeric(df["Body Mass Index"], errors="coerce")
    df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

    yes_no_map = {"Yes": 1, "No": 0}
    binary_cols = [
        "High Blood Pressure",
        "Told High Cholesterol",
        "Cholesterol Checked",
        "Smoked 100+ Cigarettes",
        "Diagnosed Stroke",
        "Diagnosed Diabetes",
        "Leisure Physical Activity",
        "Heavy Alcohol Consumption",
        "Health Care Coverage",
        "Doctor Visit Cost Barrier",
        "Difficulty Walking",
        "Vegetable or Fruit Intake (1+ per Day)"
    ]

    for col in binary_cols:
        if col in df.columns:
            df[col + "_bin"] = df[col].map(yes_no_map)

    if "Sex" in df.columns:
        df["Sex_bin"] = df["Sex"].map({"Male": 1, "Female": 0})

    df["bmi_age"] = df["Body Mass Index"] * df["Age"]
    df["bmi_squared"] = df["Body Mass Index"] ** 2

    risk_cols = [
        "High Blood Pressure_bin",
        "Told High Cholesterol_bin",
        "Smoked 100+ Cigarettes_bin",
        "Diagnosed Stroke_bin",
        "Diagnosed Diabetes_bin",
        "Difficulty Walking_bin"
    ]

    protect_cols = [
        "Leisure Physical Activity_bin",
        "Vegetable or Fruit Intake (1+ per Day)_bin"
    ]

    df["risk_flag_sum"] = df[[c for c in risk_cols if c in df.columns]].sum(axis=1)
    df["protect_flag_sum"] = df[[c for c in protect_cols if c in df.columns]].sum(axis=1)

    return df

train_fe = add_features(train)
test_fe = add_features(test)

X = train_fe.drop(columns=[TARGET])
X_test = test_fe.copy()

print("✓ สร้าง features เสร็จแล้ว")
print("X shape:", X.shape)


✓ สร้าง features เสร็จแล้ว
X shape: (221390, 36)


# Pipeline

In [ ]:
ordinal_cols = ["General Health", "Education Level", "Income Level"]

numeric_cols = [
    col for col in X.columns
    if pd.api.types.is_numeric_dtype(X[col]) and col != ID_COL
]

categorical_cols = [
    col for col in X.columns
    if col not in numeric_cols + [ID_COL] + ordinal_cols
]

ordinal_categories = [general_health_order, education_order, income_order]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("ord", ordinal_transformer, ordinal_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("✓ เตรียม pipeline เสร็จแล้ว")
print(X_train.shape, X_valid.shape)


✓ เตรียม pipeline เสร็จแล้ว
(177112, 36) (44278, 36)


# Train Model

In [ ]:
# ใช้ soft voting เพื่อให้คะแนนนิ่งกว่าใช้โมเดลเดี่ยว
log_reg = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        C=1.5,
        class_weight="balanced",
        max_iter=2000,
        solver="liblinear"
    ))
])

rf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=18,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

et = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", ExtraTreesClassifier(
        n_estimators=500,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

ensemble = VotingClassifier(
    estimators=[("lr", log_reg), ("rf", rf), ("et", et)],
    voting="soft",
    n_jobs=-1
)

ensemble.fit(X_train, y_train)
valid_proba = ensemble.predict_proba(X_valid)[:, 1]

print("✓ เทรนโมเดลเสร็จแล้ว")


✓ เทรนโมเดลเสร็จแล้ว


# Threshold Tuning

In [ ]:
def search_best_threshold(y_true, proba, beta=2.0):
    best_thr = 0.5
    best_score = -1
    rows = []

    for thr in np.arange(0.05, 0.951, 0.01):
        pred = (proba >= thr).astype(int)
        f2 = fbeta_score(y_true, pred, beta=beta)
        prec = precision_score(y_true, pred, zero_division=0)
        rec = recall_score(y_true, pred, zero_division=0)
        rows.append((thr, f2, prec, rec))

        if f2 > best_score:
            best_score = f2
            best_thr = thr

    result = pd.DataFrame(rows, columns=["threshold", "f2", "precision", "recall"])
    return best_thr, best_score, result.sort_values("f2", ascending=False)

best_thr, best_f2, threshold_table = search_best_threshold(y_valid, valid_proba, beta=2.0)

print("best threshold =", best_thr)
print("best validation F2 =", best_f2)
display(threshold_table.head(10))


best threshold = 0.4100000000000001
best validation F2 = 0.5288852725793328


,threshold,f2,precision,recall
36,0.41,0.528885,0.227309,0.791367
33,0.38,0.528752,0.217918,0.821804
34,0.39,0.528537,0.220932,0.810736
37,0.42,0.528308,0.230524,0.780299
35,0.40,0.528133,0.223866,0.799945
38,0.43,0.527255,0.233681,0.768677
32,0.37,0.526593,0.214138,0.828998
40,0.45,0.525844,0.240936,0.746541
39,0.44,0.525709,0.236771,0.756502
41,0.46,0.525277,0.244602,0.736580


# Run

In [ ]:
# เทรนด้วยข้อมูลทั้งหมด แล้วใช้ threshold ที่หาได้กับ validation
final_model = VotingClassifier(
    estimators=[("lr", log_reg), ("rf", rf), ("et", et)],
    voting="soft",
    n_jobs=-1
)

final_model.fit(X, y)
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = np.where(test_proba >= best_thr, "Yes", "No")

submission = sample_sub.copy()
submission[ID_COL] = test[ID_COL].values
submission[TARGET] = test_pred

submission.to_csv(OUTPUT_PATH, index=False)

print("✓ บันทึก submission แล้วที่", OUTPUT_PATH)
print(submission[TARGET].value_counts())
display(submission.head(10))


✓ บันทึก submission แล้วที่ /kaggle/working/submission.csv
History of HeartDisease or Attack
No     49083
Yes    25278
Name: count, dtype: int64


,ID,History of HeartDisease or Attack
0,test_000001,Yes
1,test_000002,No
2,test_000003,Yes
3,test_000004,No
4,test_000005,No
5,test_000006,Yes
6,test_000007,Yes
7,test_000008,No
8,test_000009,Yes
9,test_000010,No
